In [35]:
import pandas as pd
import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer
)
from datasets import Dataset, DatasetDict, load_dataset
from transformers import DataCollatorWithPadding
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from transformers import EarlyStoppingCallback
import joblib
import matplotlib.pyplot as plt
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [36]:
train=pd.read_csv("../outputs/cleaned_split_data/cleaned_split_train.csv")
valid=pd.read_csv("../outputs/cleaned_split_data/cleaned_split_valid.csv")
test=pd.read_csv("../outputs/cleaned_split_data/cleaned_split_test.csv")

In [37]:
print("#------Train---------")
print(train.head())
print("#------Valid---------")
print(valid.head())
print("#------Test----------")
print(test.head())

#------Train---------
                                        cleaned_text  senti_label
0      JPMorgan reels in expectations on Beyond Meat            0
1  Nomura points to bookings weakness at Carnival...            0
2  Cemex cut at Credit Suisse, J.P. Morgan on wea...            0
3                      BTIG Research cuts to Neutral            0
4            Funko slides after Piper Jaffray PT cut            0
#------Valid---------
                                        cleaned_text  senti_label
0  Monday's big rally is premature: financial adv...            1
1  Gold: Prepare For Bull Market To Begin Any Day...            2
2  Broadcom stock price target raised to $361 vs....            2
3  RECAP 12/10 +Pos Comments: + Hedgeye + Rosenbl...            2
4  Hurricane-force headwinds' pull oil lower, but...            0
#------Test----------
                                        cleaned_text  senti_label
0  London Stock Exchange : Euronext Dublin Market...            1
1  Does Th

#### Convert CSV to Datasets

In [38]:
train_dataset=Dataset.from_dict({
    "text": train["cleaned_text"].tolist(),
    "label": train["senti_label"].tolist()
})

test_dataset=Dataset.from_dict({
    "text": test["cleaned_text"].tolist(),
    "label": test["senti_label"].tolist()
})

valid_dataset=Dataset.from_dict({
    "text": valid["cleaned_text"].tolist(),
    "label": valid["senti_label"].tolist()
})

dataset=DatasetDict({
    "train": train_dataset,
    "valid": valid_dataset,
    "test": test_dataset,
})

In [39]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9532
    })
    valid: Dataset({
        features: ['text', 'label'],
        num_rows: 1193
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1192
    })
})

#### Tokenizer

In [40]:
model_id="distilbert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(model_id)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

dataset=dataset.map(tokenize, batched=True)    
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"],
    output_all_columns=True
)

Map: 100%|██████████| 1192/1192 [00:00<00:00, 58328.30 examples/s]


#### Model

In [41]:
num_labels=len(set(dataset["train"]["label"]))
model=AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels
).to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### Metrics

In [42]:
def compute_metrics(eval_pred):
    logits, labels=eval_pred
    preds=np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="macro", zero_division=0),
        "re"
        "call": recall_score(labels, preds, average="macro", zero_division=0),
        "f1": f1_score(labels, preds, average="macro", zero_division=0)

    }


#### Training Setup

In [43]:
training_args=TrainingArguments(
    output_dir="../outputs/nlp_classification_model",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    
    learning_rate=1e-5,
    weight_decay=0.05,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

#### Trainer

In [44]:
data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["valid"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

#### Train

In [45]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,2.633600,0.514709,0.823973,0.756408,0.762017,0.756859
2,0.464000,0.379506,0.865046,0.826662,0.815446,0.820358
3,0.327600,0.368372,0.870075,0.830270,0.826246,0.828234
4,0.244100,0.407638,0.872590,0.825108,0.844927,0.833836
5,0.192600,0.481342,0.865046,0.812065,0.849042,0.827698
6,0.149600,0.528334,0.870075,0.820577,0.853996,0.835594
7,0.117700,0.576989,0.876781,0.840464,0.840004,0.840183
8,0.096500,0.645320,0.869237,0.827386,0.837220,0.831983
9,0.080100,0.700452,0.865046,0.814131,0.848359,0.829798


TrainOutput(global_step=5364, training_loss=0.4784311199259349, metrics={'train_runtime': 335.2091, 'train_samples_per_second': 284.36, 'train_steps_per_second': 17.78, 'total_flos': 832096574170752.0, 'train_loss': 0.4784311199259349, 'epoch': 9.0})

#### Save and Load trained model

In [46]:
trainer.save_model("../outputs/nlp_classification_model/model")
tokenizer.save_pretrained("../outputs/nlp_classification_model/model")


('../outputs/nlp_classification_model/model\\tokenizer_config.json',
 '../outputs/nlp_classification_model/model\\special_tokens_map.json',
 '../outputs/nlp_classification_model/model\\vocab.txt',
 '../outputs/nlp_classification_model/model\\added_tokens.json',
 '../outputs/nlp_classification_model/model\\tokenizer.json')

In [47]:
tokenizer=AutoTokenizer.from_pretrained("../outputs/nlp_classification_model/model")
model=AutoModelForSequenceClassification.from_pretrained(
    "../outputs/nlp_classification_model/model"
).to(device)

#### Recreate Trainer

In [51]:
# build compute_metrics

def compute_metrics(eval_pred):
    logits, labels=eval_pred
    preds=np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="macro", zero_division=0),
        "re"
        "call": recall_score(labels, preds, average="macro", zero_division=0),
        "f1": f1_score(labels, preds, average="macro", zero_division=0)

    }
# build training_args
training_args=TrainingArguments(
    output_dir="../outputs/nlp_classification_model",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    
    learning_rate=1e-5,
    weight_decay=0.05,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

# build data_collator
data_collator=DataCollatorWithPadding(tokenizer=tokenizer)

train=pd.read_csv("../outputs/cleaned_split_data/cleaned_split_train.csv")
valid=pd.read_csv("../outputs/cleaned_split_data/cleaned_split_valid.csv")
test=pd.read_csv("../outputs/cleaned_split_data/cleaned_split_test.csv")

# build datasets instance
train_dataset=Dataset.from_dict({
    "text": train["cleaned_text"].tolist(),
    "label": train["senti_label"].tolist()
})

test_dataset=Dataset.from_dict({
    "text": test["cleaned_text"].tolist(),
    "label": test["senti_label"].tolist()
})

valid_dataset=Dataset.from_dict({
    "text": valid["cleaned_text"].tolist(),
    "label": valid["senti_label"].tolist()
})

dataset=DatasetDict({
    "train": train_dataset,
    "valid": valid_dataset,
    "test": test_dataset,
})

def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True)

tokenized_dataset=dataset.map(
    tokenize_function, batched=True
)

Map: 100%|██████████| 1192/1192 [00:00<00:00, 59859.09 examples/s]


In [54]:
trainer=Trainer(
    model=model,
    args=training_args,
    eval_dataset=tokenized_dataset["valid"],
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

#### Evaluation

In [55]:
results=trainer.evaluate()
print(pd.DataFrame(results, index=["value"]).T)

                                   value
eval_loss                       0.576989
eval_model_preparation_time     0.000000
eval_accuracy                   0.876781
eval_precision                  0.840464
eval_recall                     0.840004
eval_f1                         0.840183
eval_runtime                    1.029800
eval_samples_per_second      1158.465000
eval_steps_per_second          72.829000


#### Metrics

In [57]:
train_results=trainer.predict(tokenized_dataset["train"])
test_results=trainer.predict(tokenized_dataset["test"])

def get_metrics(pred_output):
    preds=np.argmax(pred_output.predictions, axis=1)
    labels=pred_output.label_ids
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="macro"),
        "recall": recall_score(labels, preds, average="macro"),
        "f1": f1_score(labels, preds, average="macro")
    }

train_metrics=get_metrics(train_results)
test_metrics=get_metrics(test_results)

metrics_df=pd.DataFrame([train_metrics, test_metrics], index=["Train","Test"]).T
print(metrics_df)

              Train      Test
accuracy   0.985522  0.864933
precision  0.980763  0.826908
recall     0.979662  0.811891
f1         0.980195  0.819082


#### Predictions

In [60]:
test=tokenized_dataset["test"].select(range(5))
label_encoder=joblib.load("../outputs/encoders/label_encoder.pkl")
pred_output=trainer.predict(test)


y_preds=np.argmax(pred_output.predictions,axis=1)
y_preds_labels=label_encoder.inverse_transform(y_preds)

# true labels
y_true=pred_output.label_ids
y_true_labels=label_encoder.inverse_transform(y_true)

print("Sample Predictions:")
for i in range(5):
    print(f"Text: {test['text'][i]}")
    print(f"True: {y_true_labels[i]} | Prediction: {y_preds_labels[i]}")
    print("-"*100)

Sample Predictions:
Text: London Stock Exchange : Euronext Dublin Market Notice Replacement #LondonStockExchange #Stock #MarketScreener…
True: neutral | Prediction: neutral
----------------------------------------------------------------------------------------------------
Text: Does The Teradata Corporation (NYSE:TDC) Share Price Fall With The Market?
True: neutral | Prediction: neutral
----------------------------------------------------------------------------------------------------
Text: why macro funds are shutting down left and right
True: neutral | Prediction: negative
----------------------------------------------------------------------------------------------------
Text: BDO Christmas party chaperones to guard against ghosts of scandals past
True: neutral | Prediction: neutral
----------------------------------------------------------------------------------------------------
Text: Enterprise Wins Favorable Ruling From Texas Supreme Court
True: positive | Prediction: positiv

#### Make predictions with real data

In [61]:
texts = [
    "The company reported record quarterly revenue and raised its full-year guidance.",
    "Shares fell sharply after management warned of weaker consumer demand.",
    "Revenue was largely unchanged from the previous quarter.",
    "The firm beat analysts' earnings expectations despite higher operating costs.",
    "Management expects margins to remain under pressure for the rest of the year.",
    "The company announced a new share buyback program worth $5 billion.",
    "Quarterly results were broadly in line with market expectations.",
    "Sales declined 12% as demand weakened across major markets.",
    "Free cash flow improved significantly and debt levels continued to fall.",
    "The company maintained its previous outlook for the fiscal year."
]

In [76]:
inputs=tokenizer(texts, return_tensors="pt", truncation=True, padding=True)
inputs={k:v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs=model(**inputs)

logits=outputs.logits
pred_ids=torch.argmax(logits, dim=-1).cpu().numpy()
probabilities=torch.softmax(logits, dim=1)
pred_labels=label_encoder.inverse_transform(pred_ids)
pred_probabilities=probabilities[
    torch.arange(len(texts)),
    torch.tensor(pred_ids, device=probabilities.device)
].cpu().numpy()

for t, p, s in zip(texts, pred_labels, pred_probabilities):
    print(t.strip().replace("\n", ""), "->", p, "; probability:", str(np.round(s*100,2))+"%")
    print("-"*100)

df=pd.DataFrame(
    {"Texts": texts,
     "Category": pred_labels,
     "Probabilities":[str(np.round(p*100,2))+"%" for p in pred_probabilities]}
)    

The company reported record quarterly revenue and raised its full-year guidance. -> positive ; probability: 99.75%
----------------------------------------------------------------------------------------------------
Shares fell sharply after management warned of weaker consumer demand. -> negative ; probability: 99.69%
----------------------------------------------------------------------------------------------------
Revenue was largely unchanged from the previous quarter. -> neutral ; probability: 97.04%
----------------------------------------------------------------------------------------------------
The firm beat analysts' earnings expectations despite higher operating costs. -> positive ; probability: 99.68%
----------------------------------------------------------------------------------------------------
Management expects margins to remain under pressure for the rest of the year. -> negative ; probability: 91.03%
--------------------------------------------------------------

In [78]:
pd.set_option("display.max_colwidth", None)


df

,Texts,Category,Probabilities
0,The company reported record quarterly revenue and raised its full-year guidance.,positive,99.75%
1,Shares fell sharply after management warned of weaker consumer demand.,negative,99.69%
2,Revenue was largely unchanged from the previous quarter.,neutral,97.04%
3,The firm beat analysts' earnings expectations despite higher operating costs.,positive,99.68%
4,Management expects margins to remain under pressure for the rest of the year.,negative,91.03%
5,The company announced a new share buyback program worth $5 billion.,positive,95.21%
6,Quarterly results were broadly in line with market expectations.,positive,95.16%
7,Sales declined 12% as demand weakened across major markets.,negative,95.94%
8,Free cash flow improved significantly and debt levels continued to fall.,neutral,90.41%
9,The company maintained its previous outlook for the fiscal year.,neutral,93.76%
